# What does an inadmissible heuristic actually cost you?

A* is guaranteed to return an optimal path **only** when its heuristic never overestimates
the true remaining cost. That guarantee is stated in every textbook and quietly ignored in
a lot of production pathfinding, where people scale a heuristic up to make search faster
and accept whatever comes back.

This notebook measures what that trade buys and costs, on grid maps small enough to verify
by eye:

1. **Uninformed search** — do BFS, DFS and uniform cost find the same path, at what effort?
2. **Terrain costs** — the case where BFS is confidently wrong.
3. **Admissible heuristics** — Manhattan against Chebyshev: does the stronger one expand fewer nodes?
4. **An inadmissible heuristic** — on a map built so that scaling Manhattan returns a
   *suboptimal* path, not merely a differently-searched one.

Two numbers are reported per run throughout: **cost** (was the answer right?) and
**visited** (what did it take?). Reporting only the first is how the inadmissible trap
stays hidden — the path looks fine until a map like the one in experiment 4 comes along.

The searches live in `src/gridsearch/` with no third-party dependency, and are covered by
the test suite.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from gridsearch import experiment
from gridsearch.grid import GridProblem, parse
from gridsearch.heuristics import chebyshev, check_admissible, manhattan, weighted
from gridsearch.search import astar, uniform_cost

## The maps

`T` start, `P` goal, `#` wall, a digit is the cost of *entering* that cell, anything else
costs 1.

In [ ]:
print(experiment.OPEN_MAP)
print(experiment.TERRAIN_MAP)
print(experiment.TRAP_MAP)

## Experiment 1 — uninformed search

Same map, unit costs, three strategies. BFS and uniform cost should agree; DFS has no
reason to.

In [ ]:
print(experiment.uninformed().table())

With uniform costs BFS is already optimal, so uniform cost buys nothing and pays for the
priority queue. DFS finds *a* path — read the cost column, not the fact that it returned
something.

## Experiment 2 — when cells cost different amounts

BFS minimises the **number of steps**, which stops being the same question as soon as one
cell costs more than another.

A detail worth stating, because the obvious alternative does not work: making the *move
direction* cost different amounts (left 3, right 1) cannot produce this experiment. On a
4-connected grid the net displacement fixes how many left, right, up and down moves any
path must contain, so every minimum-step route costs the same and BFS is always
cost-optimal. Terrain breaks that.

In [ ]:
print(experiment.terrain().table())

BFS takes the short route straight through the expensive cells, looking entirely healthy,
at nearly three times the cost. Uniform cost and A* agree on the cheaper way round.

A* is using Manhattan here, which counts steps and therefore *under*-estimates on a map
where a step can cost 9 — under-estimating is exactly what keeps it admissible, so the
answer stays optimal.

## Experiment 3 — admissible heuristics differ only in effort

Manhattan dominates Chebyshev on a 4-connected grid: never smaller, both lower bounds.
Dominance guarantees the stronger heuristic expands **no more** nodes — not that it
expands fewer.

In [ ]:
print(experiment.admissible_pair().table())

All three return the same cost, as they must, all being admissible. The difference is
entirely in `visited` — and on a map this small Manhattan and Chebyshev tie, which is
worth reporting rather than glossing: "no more nodes" is the actual guarantee.

## Admissibility, checked rather than assumed

The maps are small enough to compute the true remaining cost from every reachable cell by
a backwards sweep from the goal, and compare it against the heuristic. That turns
"Manhattan is admissible" from a claim into a test — and catches the case the textbook
statement hides.

In [ ]:
problem = GridProblem(parse(experiment.TERRAIN_MAP))
print("manhattan  ", check_admissible(problem, manhattan(problem.grid.goal)))
print("chebyshev  ", check_admissible(problem, chebyshev(problem.grid.goal)))
print("x3         ", check_admissible(problem, weighted(problem.grid.goal, 3.0)))

# Manhattan is admissible *on unit costs*. Halve every step cost and the
# step-counting heuristic starts overestimating.
cheap = GridProblem(parse("#######\n#T   P#\n#######"), default_cost=0.5)
print("half-cost  ", check_admissible(cheap, manhattan(cheap.grid.goal)))

## Experiment 4 — the inadmissible case

Scaling Manhattan by a constant > 1 makes A* greedier: it trusts the estimate, commits to
whatever looks closest, and stops guaranteeing optimality.

On most maps you get away with it. This map is built so you do not.

In [ ]:
print(experiment.inadmissible().table())

In [ ]:
grid = parse(experiment.TRAP_MAP)
p = GridProblem(grid)
print("optimal:")
print(grid.render(list(uniform_cost(p).path)))
print("\nManhattan x3:")
print(grid.render(list(astar(p, weighted(grid.goal, 3.0)).path)))

Both are complete, valid, plausible paths. Nothing about the second announces that a
cheaper one existed — which is exactly why the uniform-cost ground-truth row is in the
table.

And note the `visited` column: the wrong answer was found by expanding **fewer** nodes
than the right one. If weighting were only worse, nobody would do it.

## The bound

With weight *w*, the returned path costs at most *w* times optimal. That bound is what
turns weighting from an accident into a decision.

In [ ]:
for w in (1.0, 1.5, 2.0, 3.0, 5.0):
    print(experiment.bound_check(weight=w))
    print()

## Conclusion

The four experiments separate three claims that are easy to conflate:

- **Optimality is a property of the heuristic, not of A\*.** The same algorithm returns an
  optimal path with Manhattan and a worse one with Manhattan x3.
- **A better heuristic is worth having for search effort, not for answer quality.**
  Manhattan, Chebyshev and zero return identical costs; only `visited` separates them.
- **The failure is silent.** The inadmissible run returns a complete, valid path, faster.

The practical reading: weighting a heuristic is legitimate when you have decided that a
bounded amount of suboptimality is worth the speed — the bound is worth knowing, because
it makes the trade a decision. What is not defensible is scaling the heuristic to make
search faster and continuing to describe the result as shortest.